# Stage 2 — Instruction Fine-Tuning (SFT) with Unsloth

**Base model:** TinyLlama-1.1B (4-bit / QLoRA) · continues from the Stage-1 domain adapter if available.

We now teach the model **how to answer** customer-support questions using
100+ instruction/response pairs. Prompts use a simple Alpaca-style template.

**Input:** `data/instruction_dataset.jsonl`
**Output:** LoRA adapter saved to `models/sft_adapter/`

In [ ]:
# --- Environment setup (Google Colab, T4/A100 GPU) -----------------------
# Unsloth makes TinyLlama fine-tuning fit comfortably in a free Colab T4.
%%capture
# Install Unsloth and let it pull compatible recent transformers/trl/peft.
# (These notebooks use the CURRENT trl API: SFTConfig + processing_class.)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade trl peft accelerate bitsandbytes datasets
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# --- Get the project code + datasets into Colab -------------------------
# Option A: clone your GitHub repo (recommended once you have published it)
#   !git clone https://github.com/<your-username>/customer-support-assistant.git
#   %cd customer-support-assistant
#
# Option B: mount Google Drive and point PROJECT at your uploaded folder
#   from google.colab import drive; drive.mount('/content/drive')
#   PROJECT = '/content/drive/MyDrive/customer-support-assistant'
#
# For a local machine, just run the notebook from the project root.
import os
PROJECT = os.environ.get("PROJECT", ".")
DATA = os.path.join(PROJECT, "data")
MODELS = os.path.join(PROJECT, "models")
os.makedirs(MODELS, exist_ok=True)
print("PROJECT:", os.path.abspath(PROJECT))

## 1. Load model — from base, then optionally load Stage-1 domain adapter

In [ ]:
from unsloth import FastLanguageModel
import torch, os

MAX_SEQ_LENGTH = 2048
BASE_MODEL = "unsloth/tinyllama"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

# OPTION: warm-start from the Stage-1 domain-adapted adapter.
# Comment this block out to fine-tune instructions directly on the base model.
stage1 = os.path.join(MODELS, "non_instruct_adapter")
if os.path.isdir(stage1) and os.path.exists(os.path.join(stage1, "adapter_config.json")):
    print("Loading Stage-1 domain adapter as starting point...")
    model.load_adapter(stage1, adapter_name="default")
else:
    print("Stage-1 adapter not found - training instructions on base model.")

## 2. Attach / ensure LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 3. Load instruction dataset and apply the prompt template

In [ ]:
from datasets import load_dataset

PROMPT = """### Instruction:
{instruction}

### Response:
{response}"""

EOS = tokenizer.eos_token
def format_row(ex):
    return {"text": PROMPT.format(instruction=ex["instruction"],
                                  response=ex["response"]) + EOS}

jsonl = os.path.join(DATA, "instruction_dataset.jsonl")
ds = load_dataset("json", data_files=jsonl, split="train")
ds = ds.map(format_row)
print("examples:", len(ds))
print(ds[0]["text"])

## 4. Train (supervised fine-tuning)

In [ ]:
from trl import SFTTrainer, SFTConfig

# NOTE (API compatibility): recent transformers renamed Trainer's `tokenizer`
# arg to `processing_class`, and recent trl moved dataset/text settings into
# SFTConfig. This cell uses that current API.
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,      # was: tokenizer = tokenizer
    train_dataset = ds,
    args = SFTConfig(                  # was: TrainingArguments
        dataset_text_field = "text",   # moved into SFTConfig
        max_seq_length = MAX_SEQ_LENGTH,   # if error: rename to max_length
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_stage2",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()
print(trainer_stats)

## 5. Test the instruction-tuned model on domain questions

In [ ]:
FastLanguageModel.for_inference(model)

def answer(q, max_new_tokens=160):
    text = f"### Instruction:\n{q}\n\n### Response:\n"
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.9, do_sample=True)
    full = tokenizer.batch_decode(out, skip_special_tokens=True)[0]
    return full.split("### Response:")[-1].strip()

for q in ["How can I cancel my order?",
          "I received a damaged item. What should I do?",
          "How do I reset my password?"]:
    print("Q:", q); print("A:", answer(q)); print("-"*70)

## 6. Save the SFT adapter

In [ ]:
save_dir = os.path.join(MODELS, "sft_adapter")
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved SFT adapter to:", save_dir)

### Stage 2 done
The model now answers support questions in a structured, domain-specific way.
Fill in `reports/sft_model_comparison.md` with base-vs-SFT outputs, then move to
**Stage 3 (DPO alignment)**.